### — Imports

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
import pandas as pd
import plotly.express as px
from pathlib import Path
from dotenv import load_dotenv

sys.path.append(str(Path().resolve().parent))
from src.scraper import scrape_all_cities

load_dotenv(dotenv_path=Path().resolve().parent / ".env")
print("Imports OK")

### — Chargement des villes (output notebook 01)

In [ ]:
df_cities = pd.read_csv(Path().resolve().parent / "data" / "raw" / "cities.csv")
print(f"{len(df_cities)} villes chargées")
df_cities[["city_id", "city", "weather_score", "rank"]].head()

### — Paramètres

In [ ]:
CHECKIN             = "2026-06-25"
CHECKOUT            = "2026-06-26"
MAX_HOTELS_PER_CITY = 20

### — Test sur 2 villes 

df_test = scrape_all_cities(
    df_cities.head(2),
    max_hotels_per_city=3,
    headless=False,        # Chrome visible pour vérifier
    force_refresh=True     # ignore le cache pour le test
)
df_test

###  — Scraping complet 

In [ ]:
# Cellule 5 — Scraping complet 35 villes
df_hotels = scrape_all_cities(
    df_cities,
    max_hotels_per_city=20,
    headless=True,
    delay_range=(2, 4),
    force_refresh=True
)

print(f"Total : {len(df_hotels)} hôtels")
print(f"\nValeurs manquantes :\n{df_hotels.isnull().sum()}")
df_hotels.head(3)

### — Sauvegarde 

### — Nettoyage rapide avant sauvegarde finale

In [ ]:
# suppression ligne dont nom manquant

df_hotels = df_hotels.dropna(subset=["hotel_name"]).reset_index(drop=True)
print(f"Après nettoyage : {df_hotels.shape}")
df_hotels.isnull().sum()

### Sauvegarde finale de la version nettoyée :

In [ ]:
out = Path().resolve().parent / "data" / "raw" / "hotels.csv"
df_hotels.to_csv(out, index=False, encoding="utf-8-sig")
print(f"Sauvegardé : {out}")

In [ ]:
df_hotels.isnull().sum()

In [ ]:
df_hotels[df_hotels["score"].isna()][
    ["city", "hotel_name", "distance"]
]